In [2]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import gyrointerp
from gyrointerp import gyro_age_posterior
from gyrointerp import get_summary_statistics

targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\ASTR502_Mega_Target_List.csv")

In [3]:
#convert txt file to csv
tess_targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\FINAL_TRUE_TESS_Rotation_Period_Sample.csv")

print(tess_targets.head())

     ticid   periods
0   670036  5.405904
1   677945  5.956564
2  1003831  9.195566
3  1003831  9.963752
4  1003831  9.292505


In [4]:
#create an array of the TESS target TIC IDs
tess_target_tic_ids = tess_targets['ticid'].values
tess_target_periods = tess_targets['periods'].values

In [5]:
if 'mission_source' in targets.columns:
    print(f"\nMission sources in dataset:")
    print(targets['mission_source'].value_counts())
    
    # Filter for only K2 targets
    tess_targets_mega = targets[targets['mission_source'] == 'TESS'].copy()
    print(f"\nFound {len(tess_targets_mega)} K2 targets!")

    # Clean tic_id: remove leading "TIC " if present and convert to numeric (coerce failures to NA)
    tess_targets_mega['tic_id'] = tess_targets_mega['tic_id'].astype(str).str.replace('TIC ', '', regex=False)
    tess_targets_mega['tic_id'] = pd.to_numeric(tess_targets_mega['tic_id'], errors='coerce').astype('Int64')
    print(tess_targets_mega['tic_id'].head())

    # If you have a list/array of TIC IDs to match, filter to those; otherwise keep all TESS targets
    if 'tess_target_tic_ids' in globals():
        # ensure tess_target_tic_ids are integers
        try:
            tic_list = [int(x) for x in tess_target_tic_ids]
        except Exception:
            tic_list = list(tess_target_tic_ids)
        matched = tess_targets[tess_targets['ticid'].isin(tic_list)].copy()
        matched_teff = tess_targets_mega[tess_targets_mega['tic_id'].isin(tic_list)].copy()
        print(f"\nMatched {len(matched)} TESS targets from provided TIC ID list.")
    else:
        matched = tess_targets.copy()
        print("\nNo external TIC ID list found; using all TESS targets.")

    # Build tess_star_df with columns required downstream: 'target_name', 'tic_id', 'Teff', 'period'
    target_results = []
    for idx, r in matched.iterrows():
        tic = int(r['ticid']) if pd.notnull(r['ticid']) else None
        target_name = r.get('pl_name') or r.get('hostname') or f"TIC{tic}"
        #have to get teff from the mega target list
        teff = matched_teff[matched_teff['tic_id'] == tic]['st_teff'].values
        lit_age = matched_teff[matched_teff['tic_id'] == tic]['st_age'].values

        #want to find the single period value for this tic id
        period = tess_targets['periods'][tess_targets['ticid'] == tic].values

        target_results.append({
            'tic_ids': target_name,
            'Teff': teff,
            'period': period,
            'st_age': lit_age
        })

    # create DataFrame even if empty so later cells won't raise NameError
    tess_star_df = pd.DataFrame(target_results, columns=['tic_ids', 'Teff', 'period', 'st_age'])
    print(f"\nFinal tess_star_df has {len(tess_star_df)} rows.")

else:
    # If no 'mission_source' column, create empty tess_star_df to avoid NameError later
    print("No 'mission_source' in targets DataFrame; creating empty tess_star_df.")
    tess_star_df = pd.DataFrame(columns=['target_name', 'tic_ids', 'Teff', 'period'])
    print(tess_star_df.head())


Mission sources in dataset:
mission_source
Kepler    2762
TESS       717
K2         548
WASP       168
HAT        139
Other      105
CoRoT       34
NGTS        22
KELT        21
Name: count, dtype: int64

Found 717 K2 targets!
0     201248411
5      12421862
7      52005579
10    394137592
11    327369524
Name: tic_id, dtype: Int64

Matched 1293 TESS targets from provided TIC ID list.

Final tess_star_df has 1293 rows.


In [6]:
Teff = tess_star_df['Teff']
print(Teff.head())
Prot = tess_star_df['period']
print(Prot.head())
lit_age = tess_star_df['st_age']
print(lit_age.head())

0          []
1          []
2    [5640.0]
3    [5640.0]
4    [5640.0]
Name: Teff, dtype: object
0                             [5.405904436]
1                              [5.95656414]
2    [9.19556627, 9.963752119, 9.292504661]
3    [9.19556627, 9.963752119, 9.292504661]
4    [9.19556627, 9.963752119, 9.292504661]
Name: period, dtype: object
0       []
1       []
2    [7.3]
3    [7.3]
4    [7.3]
Name: st_age, dtype: object


In [7]:
print(tess_star_df)

           tic_ids      Teff  \
0        TIC670036        []   
1        TIC677945        []   
2       TIC1003831  [5640.0]   
3       TIC1003831  [5640.0]   
4       TIC1003831  [5640.0]   
...            ...       ...   
1288  TIC466884459        []   
1289  TIC466884459        []   
1290  TIC466884459        []   
1291  TIC466884459        []   
1292  TIC468989066        []   

                                                 period st_age  
0                                         [5.405904436]     []  
1                                          [5.95656414]     []  
2                [9.19556627, 9.963752119, 9.292504661]  [7.3]  
3                [9.19556627, 9.963752119, 9.292504661]  [7.3]  
4                [9.19556627, 9.963752119, 9.292504661]  [7.3]  
...                                                 ...    ...  
1288  [5.921955847, 5.799189828, 5.671638143, 5.8211...     []  
1289  [5.921955847, 5.799189828, 5.671638143, 5.8211...     []  
1290  [5.921955847, 5.79918982

In [ ]:
# calculate dictionary of summary statistics for each target and store results in results_df
# Note: gyro_age_posterior and get_summary_statistics were imported in earlier cells,
# so we don't re-import them here.

# ensure columns exist (store arrays as objects)
for col in ['age_grid', 'age_posterior', 'median', '+1sigma', '-1sigma', 'mean', 'mode']:
    if col not in tess_star_df.columns:
        tess_star_df[col] = [None] * len(tess_star_df)

for i in range(tess_star_df.shape[0]):
    Prot = np.mean(tess_star_df['period'].iloc[i])
    Prot_err = 0.2

    Teff = np.mean(tess_star_df['Teff'].iloc[i])
    Teff_err = 100

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    # compute summary statistics
    result = get_summary_statistics(age_grid, age_posterior)

    # store results in the dataframe
    tess_star_df.at[i, 'age_grid'] = age_grid
    tess_star_df.at[i, 'age_posterior'] = age_posterior
    tess_star_df.at[i, 'median'] = result.get('median', np.nan)
    tess_star_df.at[i, '+1sigma'] = result.get('+1sigma', np.nan)
    tess_star_df.at[i, '-1sigma'] = result.get('-1sigma', np.nan)
    tess_star_df.at[i, 'mean'] = result.get('mean', np.nan)
    tess_star_df.at[i, 'mode'] = result.get('mode', np.nan)

    print(f"\nTarget: {tess_star_df['tic_ids'].iloc[i]}")
    print(f"Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Target: TIC670036
Age = nan +nan -nan Myr.


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Target: TIC677945
Age = nan +nan -nan Myr.

Target: TIC1003831
Age = 805.66 +102.76 -84.66 Myr.

Target: TIC1003831
Age = 805.66 +102.76 -84.66 Myr.

Target: TIC1003831
Age = 805.66 +102.76 -84.66 Myr.


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


...

c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Target: TIC137817355
Age = nan +nan -nan Myr.


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Target: TIC137817355
Age = nan +nan -nan Myr.


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
#match these with st_age (ages from the literature) to see how well they compare 
tess_star_df  = tess_star_df.rename(columns={'tic_id': 'tic_ids'})
#k2_star_df = k2_star_df.astype({'tic_id': np.int64})
#k2_targets_mega = k2_targets_mega.astype({'tic_id': np.int64})
#k2_star_df = k2_star_df.merge(k2_targets_mega[["tic_id", "st_age"]], on="tic_id", how="left")


for i in range(len(tess_star_df)):
    lit_age = tess_star_df['st_age'].iloc[i]

    # Determine whether lit_age contains a usable value.
    # lit_age may be a scalar (float / pd.NA) or an array-like (np.ndarray, list, Series).
    if hasattr(lit_age, '__len__') and not np.isscalar(lit_age):
        # array-like: check length and that not all entries are NaN
        has_value = len(lit_age) > 0 and not np.all(pd.isna(lit_age))
        lit_age_repr = np.array2string(lit_age)
    else:
        # scalar: check for NaN / missing
        has_value = not pd.isna(lit_age)
        lit_age_repr = str(lit_age)

    if has_value:
        tic_col = 'tic_ids' if 'tic_ids' in tess_star_df.columns else 'tic_id'
        derived_age = tess_star_df['median'].iloc[i]
        age_uncertainty = tess_star_df['+1sigma'].iloc[i], tess_star_df['-1sigma'].iloc[i]
        print(f"Target: {tess_star_df[tic_col].iloc[i]}, Literature Age: {lit_age_repr} Gyr, Derived Age with Uncertainty: {derived_age} Myr ± {age_uncertainty[0]} Myr / {age_uncertainty[1]} Myr")

In [ ]:
import matplotlib.pyplot as plt

for i in range(len(tess_star_df)):
    age_grid = tess_star_df['age_grid'][i]
    age_posterior = tess_star_df['age_posterior'][i]
    fig, ax = plt.subplots()
    ax.plot(age_grid, 1e3*age_posterior, c='k', lw=1)
    ax.update({
        'xlabel': 'Age [Myr]',
        'ylabel': 'Probability ($10^{-3}\,$Myr$^{-1}$)',
        'title': f'Prot = {Prot}d, Teff = {Teff}K',
        'xlim': [0,4000]
    })
    plt.show()